In [1]:
# 04_baf_baseline — Cell 1: load clean BAF, chronological split (months 0-4 / 5 / 6-7), save split summary
import sys, json; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
from src.config import PATHS, SEED, Timer

with Timer("read BAF parquet"):
    baf = pd.read_parquet(PATHS["work"] / "baf_base_clean.parquet")

TARGET = "fraud_bool"
parts = {"train": baf[baf["month"] <= 4],
         "valid": baf[baf["month"] == 5],
         "test":  baf[baf["month"] >= 6]}
summary_split = {k: {"months": sorted(map(int, d["month"].unique())), "n": int(len(d)),
                     "n_fraud": int(d[TARGET].sum()), "prevalence": round(float(d[TARGET].mean()), 5)}
                 for k, d in parts.items()}
for k, v in summary_split.items():
    print(k, v)

assert sum(len(d) for d in parts.values()) == len(baf)
assert parts["train"]["month"].max() < parts["valid"]["month"].min() < parts["test"]["month"].min()
(PATHS["repo"] / "reports" / "split_baf.json").write_text(json.dumps(summary_split, indent=2))
print("\nsplit OK: no month overlap, no row lost; saved reports/split_baf.json")

[read BAF parquet] wall = 3.5 s | RSS now = 0.61 GB | peak RSS = 0.61 GB
train {'months': [0, 1, 2, 3, 4], 'n': 675666, 'n_fraud': 6740, 'prevalence': 0.00998}
valid {'months': [5], 'n': 119323, 'n_fraud': 1411, 'prevalence': 0.01183}
test {'months': [6, 7], 'n': 205011, 'n_fraud': 2878, 'prevalence': 0.01404}

split OK: no month overlap, no row lost; saved reports/split_baf.json


In [2]:
%%writefile ../src/metrics.py
"""Ranking metrics shared by all notebooks (BAF and AML). Higher score = more suspicious."""
import numpy as np
from sklearn.metrics import average_precision_score, roc_curve


def pr_auc(y, s):
    return float(average_precision_score(y, s))


def recall_at_fpr(y, s, fpr_target=0.05):
    """Best recall (TPR) reachable while FPR <= fpr_target.
    With few negatives the FPR grid is coarse (step = 1/n_neg); meaningful on full splits."""
    fpr, tpr, _ = roc_curve(y, s)
    return float(tpr[fpr <= fpr_target].max())


def threshold_at_fpr(y, s, fpr_target=0.05):
    """Score cut-off that flags fpr_target of the negatives.
    Use on VALIDATION to pick an operating threshold; apply unchanged on test."""
    return float(np.quantile(np.asarray(s)[np.asarray(y) == 0], 1 - fpr_target))


def precision_at_k(y, s, k=100):
    top = np.argsort(-np.asarray(s), kind="stable")[:k]
    return float(np.asarray(y)[top].mean())


def lift_at_frac(y, s, frac=0.01):
    k = max(1, int(len(y) * frac))
    return precision_at_k(y, s, k) / float(np.mean(y))


def summary(y, s, ks=(100, 500)):
    out = {"prevalence": float(np.mean(y)), "pr_auc": pr_auc(y, s),
           "recall_at_5fpr": recall_at_fpr(y, s, 0.05), "lift_at_1pct": lift_at_frac(y, s, 0.01)}
    for k in ks:
        out[f"precision_at_{k}"] = precision_at_k(y, s, k)
    return out

Writing ../src/metrics.py


In [3]:
# 04_baf_baseline — Cell 3: test the metric functions against hand-computed values
import importlib, src.metrics as M
importlib.reload(M)

y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])
s = np.array([.1, .2, .3, .4, .5, .6, .7, .9, .8, .95])   # one negative (0.9) outranks a positive
got = M.summary(y, s, ks=(1, 3))
got["threshold_at_fpr_0.125"] = M.threshold_at_fpr(y, s, 0.125)
expected = {"prevalence": 0.2, "pr_auc": 0.8333, "recall_at_5fpr": 0.5, "lift_at_1pct": 5.0,
            "precision_at_1": 1.0, "precision_at_3": 0.6667, "threshold_at_fpr_0.125": 0.725}
for k, e in expected.items():
    ok = abs(got[k] - e) < 1e-3
    print(f"{k:24s} got={got[k]:.4f}  expected={e:.4f}  {'OK' if ok else 'MISMATCH'}")
    assert ok, k
print("\nALL METRIC TESTS PASSED")

prevalence               got=0.2000  expected=0.2000  OK
pr_auc                   got=0.8333  expected=0.8333  OK
recall_at_5fpr           got=0.5000  expected=0.5000  OK
lift_at_1pct             got=5.0000  expected=5.0000  OK
precision_at_1           got=1.0000  expected=1.0000  OK
precision_at_3           got=0.6667  expected=0.6667  OK
threshold_at_fpr_0.125   got=0.7250  expected=0.7250  OK

ALL METRIC TESTS PASSED


In [4]:
# One-off: append DECISION #8 to decisions.md
from pathlib import Path

TEXT = """

## DECISION #8 — BAF baseline: features, model, procedure (2026-09-27, before any training)
- Order: (1) DECISION #5 A/B/C compared on validation (month 5); (2) the winner's model
  from that round is the final model, no retraining (deterministic, same seed);
  (3) test (months 6-7) evaluated once. Month 5 serves both early stopping and the
  A/B/C choice: one validation split, not independent confirmation.
- Features: all raw columns except fraud_bool and month (month = absolute time index):
  29 raw. 5 categorical columns (identified in 02_audit_baf) as native LightGBM
  category. Flags <col>_is_missing (v < 0) for the 5 columns coded with -1, in every
  variant. credit_risk_score and velocity_6h have real negatives: kept raw, no flag.
  intended_balcon_amount: A = raw (34 features); B = raw + flag (35);
  C = NaN when v < 0 + flag (35). No engineered features in the baseline.
- Model: lightgbm.train (native API), LightGBM 4.7.0, objective=binary,
  learning_rate=0.05, num_leaves=63, min_child_samples=100, feature_fraction=0.8,
  bagging_fraction=0.8, bagging_freq=1, up to 2000 rounds, early stopping 100 rounds
  on validation average_precision; best_iteration reported.
- Imbalance: no class weighting. BAF paper settings are not replicated or claimed.
- Reproducibility: seed=42, deterministic=True, force_col_wise=True, num_threads=8.
- Early-stopping metric = LightGBM average_precision; reported PR-AUC = sklearn
  average_precision_score (small differences possible, reported if > 1e-3).
- Reported on test: src.metrics.summary (PR-AUC, Recall@5%FPR, Precision@100/500,
  Lift@1%) and the FPR ratio customer_age >= 50 vs < 50 at the 5%-FPR threshold
  chosen on validation (group cut-off 50 from the handoff, section 5.1). Report only.
"""

p = Path("../reports/decisions.md")
assert "DECISION #8 " not in p.read_text(encoding="utf-8"), "#8 already present"
with open(p, "a", encoding="utf-8") as f:
    f.write(TEXT)
print("written")

written
